# 05 — XGBoost Model

**Workstream**: Modeling — XGBoost  ·  **Owner**: Deepak (backup: Bella)  ·  **Last touched**: 2026-06-02

**What this notebook decides**

Train and evaluate a calibrated XGBoost classifier on the **same feature
contract** and **same temporal split** as notebook 04 (the LogReg baseline).
Direct A/B comparison.

Per CLAUDE.md, XGBoost is the production estimator iff it clears the
baseline on **both**:
  - **PR-AUC** (headline metric)
  - **Precision@10%** (operational metric)

Baseline target to beat (current, from `reports/metrics/baseline_[0-9]*.json`
on the refreshed data): PR-AUC test ~0.256, Precision@10% test ~30%. These move
with the data snapshot — the notebook reads the latest baseline at runtime.

**Why XGBoost should win**
  - Captures non-linear interactions LogReg can't (e.g. high `prior_fails`
    *and* short `days_since_last_inspection` *and* a `flag_kw_rodent` hit —
    an interaction term LogReg has no access to).
  - Handles missing values natively, retaining the "first inspection at
    license" signal that median-imputation in LogReg destroys.
  - Partition-based categorical splits avoid the one-hot explosion of
    `static_zip` (~60 levels).

Per CLAUDE.md: `scale_pos_weight` (not SMOTE). Class-imbalance handling
via loss re-weighting only.

**Deliverables** (filenames carry the run id = `<date>_<short-git-sha>`)
- `data/models/xgb_<run_id>.joblib` — calibrated estimator
- `data/models/xgb_<run_id>_metadata.json` — split cutoffs, hyperparameters, metrics + Tier-0 provenance
- `reports/metrics/xgb_<run_id>.json` — git-tracked metrics report

## 1. Setup

In [ ]:
import sys
import json
from datetime import date
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

from foodsafety.config import MODELS_DIR, PROCESSED_DIR, RANDOM_STATE
from foodsafety.utils.time import temporal_split, summarize, expanding_year_folds
from foodsafety.models.baseline import ALL_FEATURES, LABEL_COL
from foodsafety.models.xgb import (
    build_xgb_estimator, prepare_xgb_features,
    extract_categorical_dtypes, compute_scale_pos_weight,
)
from foodsafety.models.evaluate import (
    evaluate, decile_lift_table, calibration_table,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Load + split (same cutoffs as 04)

Re-use the cutoffs from the baseline notebook. If they ever change there,
they must change here too. Wiring a shared constant would help; keeping them
as a literal here for now so the diff is visible across notebooks.

In [ ]:
features_path = PROCESSED_DIR / 'features.parquet'
features = pd.read_parquet(features_path)
features['inspection_date'] = pd.to_datetime(features['inspection_date'])
for c in features.columns:
    if c.startswith('flag_kw_'):
        features[c] = features[c].astype('int8')

TRAIN_END = '2024-07-01'
VAL_END   = '2025-07-01'

split = temporal_split(features, train_end=TRAIN_END, val_end=VAL_END)
for name, frame in [('train', split.train), ('val', split.val), ('test', split.test)]:
    s = summarize(frame, label_col=LABEL_COL)
    print(f'  {name:<5}  n={s.rows:>6,}  '
          f'dates {s.date_min.date()} → {s.date_max.date()}  '
          f'positive_rate={s.positive_rate:.2%}')

## 3. Prepare features for XGBoost

XGBoost reads pandas `category` dtype natively when `enable_categorical=True`.
Critical: the category levels must align across train/val/test so the
internal codes match. We infer levels from train, then reuse them on val
and test.

In [ ]:
X_train_raw = split.train[ALL_FEATURES]
y_train     = split.train[LABEL_COL].astype(int)
X_val_raw   = split.val  [ALL_FEATURES]
y_val       = split.val  [LABEL_COL].astype(int)
X_test_raw  = split.test [ALL_FEATURES]
y_test      = split.test [LABEL_COL].astype(int)

X_train = prepare_xgb_features(X_train_raw)
cat_dtypes = extract_categorical_dtypes(X_train)
X_val  = prepare_xgb_features(X_val_raw,  categorical_dtypes=cat_dtypes)
X_test = prepare_xgb_features(X_test_raw, categorical_dtypes=cat_dtypes)

# --- Early-stopping holdout carved from the TAIL of train -------------------
# Double-dip fix: val is reserved for calibration only, so early stopping gets
# its own held-out set — the last ~6 months of train (`train_es`). A 180-day
# embargo gap is dropped between the fit set and train_es: the label is
# *_next_180d, so a fit anchor within 180d of the boundary would "see" outcomes
# that land inside train_es and leak the early-stopping signal.
ES_START = pd.Timestamp('2024-01-01')          # last ~6 months of train
EMBARGO  = pd.Timedelta(days=180)
train_dates = split.train['inspection_date']
fit_mask = (train_dates < (ES_START - EMBARGO)).to_numpy()
es_mask  = (train_dates >= ES_START).to_numpy()
X_train_fit, y_train_fit = X_train[fit_mask], y_train[fit_mask]
X_train_es,  y_train_es  = X_train[es_mask],  y_train[es_mask]
print(f'fit set   n={len(X_train_fit):>6,}  (date < {(ES_START - EMBARGO).date()})')
print(f'es  set   n={len(X_train_es):>6,}  (date >= {ES_START.date()})  '
      f'[{int(EMBARGO.days)}d embargo gap dropped]')

print('\nTrain dtypes after prep:')
print(X_train.dtypes.value_counts())

# scale_pos_weight on FULL train — that is what the final model is refit on.
spw = compute_scale_pos_weight(y_train)
print(f'\nscale_pos_weight (n_neg/n_pos in train) = {spw:.3f}')

## 4. Fit with early stopping

**Double-dip fix (this iteration).** Previously `val` was used twice — for
early stopping *and* for calibration — which optimistically biases both the
chosen tree count and the calibration map. Now:

  - Early stopping runs against a held-out **tail of train** (`train_es`, the
    last ~6 months), with a **180-day embargo** gap dropped before it so a
    fit-set anchor's `*_next_180d` label can't peek into `train_es`.
  - We find `best_iteration` on that embargoed fit/es split, then **refit on
    the full train** at that fixed tree count — no train data wasted on a
    permanent holdout.
  - **`val` is reserved for calibration only.** Test stays fully held out.

In [ ]:
%%time
# Probe fit: train on the embargoed fit-set, early-stop on the held-out tail
# (train_es). This finds best_iteration WITHOUT touching val, which stays pure
# for calibration.
probe = build_xgb_estimator(scale_pos_weight=spw, early_stopping_rounds=40)
probe.fit(X_train_fit, y_train_fit, eval_set=[(X_train_es, y_train_es)], verbose=False)
probe_best_iter  = int(probe.best_iteration)
probe_best_score = float(probe.best_score)
print(f'Best iteration (early-stop on train_es): {probe_best_iter}')
print(f'Best train_es aucpr:                     {probe_best_score:.4f}')

# Refit on FULL train at the fixed tree count (early stopping disabled) so the
# production model uses all of train — no data permanently held out.
# best_iteration is 0-indexed, so the tree count is best_iteration + 1.
N_TREES = probe_best_iter + 1
xgb = build_xgb_estimator(
    scale_pos_weight=spw,
    n_estimators=N_TREES,
    early_stopping_rounds=None,
)
xgb.fit(X_train, y_train, verbose=False)
print(f'Refit on full train ({len(X_train):,} rows) with {N_TREES} trees.')

In [ ]:
%%time
# --- Expanding-window time-series CV on the TRAIN subset only ---------------
# Two questions the single train->val score can't answer on its own:
#   (a) does the configured XGB generalise across years, or is one val score a
#       fluke?  -> mean +/- std fold PR-AUC
#   (b) sigmoid vs isotonic calibration -> which wins on held-out years?
# Folds are by calendar year with a 180-day embargo (the *_next_180d label
# needs it; quarterly folds are too small at this ~10% prevalence). PR-AUC is
# rank-based, so the raw (uncalibrated) model answers (a); for (b) we calibrate
# on an embargoed tail of each fold's own train and score Brier on its val year.
folds = expanding_year_folds(split.train, embargo_days=180)
train_dates_all = split.train['inspection_date'].reset_index(drop=True)

fold_rows = []
for k, (tr_idx, va_idx) in enumerate(folds, 1):
    Xtr, ytr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
    Xva, yva = X_train.iloc[va_idx], y_train.iloc[va_idx]
    val_year = int(train_dates_all.iloc[va_idx].dt.year.iloc[0])

    m = build_xgb_estimator(scale_pos_weight=spw, n_estimators=N_TREES,
                            early_stopping_rounds=None)
    m.fit(Xtr, ytr, verbose=False)
    pr_auc = evaluate(yva.to_numpy(), m.predict_proba(Xva)[:, 1]).to_dict()['pr_auc']

    # (b) embargoed base/calibration split WITHIN the fold's train.
    tr_dates = train_dates_all.iloc[tr_idx]
    cal_start = tr_dates.sort_values().iloc[int(0.85 * len(tr_dates))]
    base_m = (tr_dates < (cal_start - pd.Timedelta(days=180))).to_numpy()
    cal_m  = (tr_dates >= cal_start).to_numpy()
    briers = {}
    if base_m.sum() > 100 and cal_m.sum() > 50:
        b = build_xgb_estimator(scale_pos_weight=spw, n_estimators=N_TREES,
                                early_stopping_rounds=None)
        b.fit(Xtr[base_m], ytr[base_m], verbose=False)
        for method in ('sigmoid', 'isotonic'):
            cc = CalibratedClassifierCV(FrozenEstimator(b), method=method)
            cc.fit(Xtr[cal_m], ytr[cal_m])
            briers[method] = evaluate(
                yva.to_numpy(), cc.predict_proba(Xva)[:, 1]
            ).to_dict()['brier_score']
    fold_rows.append({'fold': k, 'val_year': val_year, 'n_val': len(va_idx),
                      'pr_auc': pr_auc,
                      'brier_sigmoid': briers.get('sigmoid'),
                      'brier_isotonic': briers.get('isotonic')})

cv = pd.DataFrame(fold_rows)
print(cv.round(4).to_string(index=False))
print(f'\nconfig PR-AUC across folds: mean={cv.pr_auc.mean():.4f}  std={cv.pr_auc.std():.4f}')

mean_sig = cv['brier_sigmoid'].mean()
mean_iso = cv['brier_isotonic'].mean()
print(f'mean Brier  sigmoid={mean_sig:.4f}  isotonic={mean_iso:.4f}  (lower is better)')
# Tie-break to sigmoid: it yields continuous scores (isotonic ties many rows to
# the same probability, which the UI ranking depends on — see the served script).
CALIB_METHOD = 'isotonic' if mean_iso < mean_sig else 'sigmoid'
print(f'-> CALIB_METHOD = {CALIB_METHOD!r}')

In [ ]:
# Calibrate on val using the method chosen by the expanding-window CV above.
# val is now used ONLY for calibration (early stopping used train_es), so this
# is no longer a double-dip. FrozenEstimator marks the base model pre-fit, so
# only the calibration mapping is learned — the trees are not refit.
model = CalibratedClassifierCV(FrozenEstimator(xgb), method=CALIB_METHOD)
model.fit(X_val, y_val)
print(f'Calibrated ({CALIB_METHOD}) on val.')

## 5. Evaluate

In [ ]:
val_scores  = model.predict_proba(X_val)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

val_report  = evaluate(y_val,  val_scores)
test_report = evaluate(y_test, test_scores)

# Load baseline test metrics for A/B comparison
baseline_reports = sorted((_PROJECT_ROOT / 'reports' / 'metrics').glob('baseline_[0-9]*.json'))
if baseline_reports:
    baseline_test = json.loads(baseline_reports[-1].read_text())['test']
else:
    baseline_test = None
    print('No baseline metrics found — skipping A/B comparison.')

out = {'val':  val_report.to_dict(),
       'test': test_report.to_dict()}
if baseline_test is not None:
    out['baseline_test'] = baseline_test
    out['delta_vs_baseline'] = {
        k: round(test_report.to_dict()[k] - baseline_test[k], 4)
        for k in ['pr_auc', 'roc_auc', 'precision_at_5pct',
                  'precision_at_10pct', 'precision_at_20pct',
                  'top_decile_lift', 'brier_score']
    }

print(pd.DataFrame(out).round(4).to_string())

### 5a. Did XGBoost beat the baseline?

CLAUDE.md requires clearing both PR-AUC and precision@10%.

In [ ]:
if baseline_test is not None:
    pr_auc_delta = test_report.pr_auc - baseline_test['pr_auc']
    p10_delta    = test_report.precision_at_10pct - baseline_test['precision_at_10pct']
    print(f'PR-AUC          : baseline {baseline_test["pr_auc"]:.4f}  →  '
          f'XGB {test_report.pr_auc:.4f}  ({pr_auc_delta:+.4f})')
    print(f'Precision@10%   : baseline {baseline_test["precision_at_10pct"]:.4f}  →  '
          f'XGB {test_report.precision_at_10pct:.4f}  ({p10_delta:+.4f})')
    if pr_auc_delta > 0 and p10_delta > 0:
        print('\n✓ XGBoost beats baseline on BOTH metrics.')
        print('  → Production estimator candidate.')
    elif pr_auc_delta > 0 or p10_delta > 0:
        print('\n△ XGBoost beats baseline on ONE metric, not both.')
        print('  → Investigate before promoting. May need tuning / better features.')
    else:
        print('\n✗ XGBoost does not beat baseline.')
        print('  → Likely a feature problem, not a model problem. Stay with baseline.')

### 5b. Decile lift (TEST)

Per-decile positive rate. Decile 1 = top 10% predicted scores.

In [ ]:
lift = decile_lift_table(y_test, test_scores)
print(lift.to_string())

### 5c. Calibration (TEST)

In [ ]:
calib = calibration_table(y_test, test_scores, n_bins=10)
print(calib.to_string())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color='#9CA3AF', linestyle=':', label='Perfect')
ax.plot(calib['mean_predicted'], calib['mean_observed'], 'o-', color='#15110D',
        label='XGBoost (test)')
ax.set_xlabel('Mean predicted'); ax.set_ylabel('Mean observed')
ax.set_title('Calibration curve — test set'); ax.legend()
lim = max(calib['mean_predicted'].max(), calib['mean_observed'].max()) * 1.1
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
plt.tight_layout(); plt.show()

### 5d. Feature importance (gain)

Top 15 features by XGBoost's gain-based importance. This is the cheap,
interpretable view — SHAP (notebook 06) is the formal one.

In [ ]:
importances = pd.Series(xgb.feature_importances_, index=ALL_FEATURES)
top15 = importances.sort_values(ascending=False).head(15)
print('Top 15 features by gain:')
print(top15.round(4).to_string())

ax = top15[::-1].plot.barh(color='#15110D', figsize=(8, 6))
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost — top 15 features')
plt.tight_layout(); plt.show()

### 5e. Correlation matrix + comparison with XGBoost gain (top 15)

Pearson correlation of numeric features against the label (train set), compared to XGBoost gain importance.
High gain + low correlation → XGBoost is exploiting non-linear structure.
High correlation + low gain → feature may be redundant or collinear with another.

In [ ]:
# --- Correlation matrix of numeric features (train set) ---
# Exclude categoricals — Pearson correlation is not defined for them.
num_features = [f for f in ALL_FEATURES if X_train[f].dtype != 'category']
train_num = X_train[num_features].copy().astype(float)

corr_matrix = train_num.corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.03)
ax.set_xticks(range(len(num_features))); ax.set_xticklabels(num_features, rotation=90, fontsize=7)
ax.set_yticks(range(len(num_features))); ax.set_yticklabels(num_features, fontsize=7)
ax.set_title('Feature correlation matrix (numeric, train set)')
plt.tight_layout(); plt.show()

# --- Correlation with label vs XGBoost gain ---
label_corr = train_num.corrwith(y_train.astype(float)).abs().rename('corr_with_label')

xgb_gain = pd.Series(xgb.feature_importances_, index=ALL_FEATURES)
xgb_gain_norm = (xgb_gain / xgb_gain.max()).rename('xgb_gain_norm')

comparison = pd.concat([label_corr, xgb_gain_norm], axis=1).dropna()
comparison = comparison.sort_values('xgb_gain_norm', ascending=False)

print("Feature comparison — |corr with label| vs normalised XGB gain (top 25):")
print(comparison.head(25).round(4).to_string())

# Side-by-side bar chart: top 15 XGB gain features that are numeric (comparable)
top15_names = [f for f in top15.index if f in comparison.index]
comp15 = comparison.loc[top15_names].sort_values('xgb_gain_norm', ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
y = range(len(comp15))
ax.barh([i - 0.2 for i in y], comp15['xgb_gain_norm'], height=0.35,
        color='#15110D', label='XGB gain (normalised)')
ax.barh([i + 0.2 for i in y], comp15['corr_with_label'], height=0.35,
        color='#D97706', label='|Pearson corr with label|')
ax.set_yticks(list(y)); ax.set_yticklabels(comp15.index, fontsize=9)
ax.set_xlabel('Score (normalised)')
ax.set_title('Top numeric features — XGB gain vs label correlation\n(categorical features excluded from correlation)')
ax.legend()
plt.tight_layout(); plt.show()

# Flag features where XGB gain >> correlation (non-linear signal)
threshold = 0.3
nl_features = comp15[comp15['xgb_gain_norm'] - comp15['corr_with_label'] > threshold]
if not nl_features.empty:
    print("\nFeatures with strong non-linear signal (gain − corr > 0.3):")
    print(nl_features[['xgb_gain_norm', 'corr_with_label']].round(4).to_string())

# Note any top-15 features that were categorical (skipped from comparison)
skipped = [f for f in top15.index if f not in comparison.index]
if skipped:
    print(f"\nCategorical features in top-15 gain (no Pearson corr): {skipped}")


## 6. Persist

Same convention as notebook 04: never overwrite; the filename carries the run id
(`<date>_<short-git-sha>`); write a metadata sidecar with Tier-0 provenance
(git SHA, `features_sha256`, `feature_set_version`).

In [ ]:
from foodsafety.tracking import provenance

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS_DIR = _PROJECT_ROOT / 'reports' / 'metrics'
REPORTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Tier-0 provenance — same block as the baseline notebook + the served script.
prov = provenance(features_path, ALL_FEATURES, _PROJECT_ROOT)
run_id = prov['run_id']
model_path = MODELS_DIR / f'xgb_{run_id}.joblib'
metadata_path = MODELS_DIR / f'xgb_{run_id}_metadata.json'
report_path = REPORTS_METRICS_DIR / f'xgb_{run_id}.json'

joblib.dump(model, model_path)
print(f'saved model → {model_path}  ({model_path.stat().st_size / 1e6:.1f} MB)')

metadata = {
    'model': f'xgboost_{CALIB_METHOD}',
    'run_id': run_id,
    'git_commit': prov['git_commit'],
    'git_dirty': prov['git_dirty'],
    'random_state': RANDOM_STATE,
    'date_trained': date.today().isoformat(),
    'calibration': f'{CALIB_METHOD} on val (FrozenEstimator prefit); '
                   'chosen by expanding-window CV (mean Brier)',
    'early_stopping': {
        # Double-dip fix: best_iteration found on an embargoed tail of train
        # (train_es), NOT on val; val is used for calibration only.
        'eval_set': 'train_es (last ~6 months of train)',
        'embargo_days': int(EMBARGO.days),
        'rounds': 40,
        'best_iteration': probe_best_iter,
        'best_es_aucpr': probe_best_score,
        'fit_set_n': int(len(X_train_fit)),
        'es_set_n': int(len(X_train_es)),
    },
    'cv': {
        'scheme': 'expanding_year_folds (180d embargo)',
        'n_folds': int(len(cv)),
        'pr_auc_mean': float(cv['pr_auc'].mean()),
        'pr_auc_std': float(cv['pr_auc'].std()),
        'mean_brier_sigmoid': float(mean_sig),
        'mean_brier_isotonic': float(mean_iso),
    },
    'split': {
        'train_end': str(split.train_end.date()),
        'val_end':   str(split.val_end.date()),
        'train_n':   int(len(split.train)),
        'val_n':     int(len(split.val)),
        'test_n':    int(len(split.test)),
    },
    'hyperparameters': {
        'n_estimators': int(N_TREES),
        'max_depth':        xgb.max_depth,
        'learning_rate':    xgb.learning_rate,
        'scale_pos_weight': float(spw),
    },
    'features': {
        'all':       ALL_FEATURES,
        'label_col': LABEL_COL,
        'n_features': len(ALL_FEATURES),
        'feature_set_version': prov['feature_set_version'],
    },
    'dataset': {
        'features_parquet': 'data/processed/features.parquet',
        'features_sha256':  prov['features_sha256'],
    },
    'metrics': {
        'val':  val_report.to_dict(),
        'test': test_report.to_dict(),
    },
}
with metadata_path.open('w') as f:
    json.dump(metadata, f, indent=2)
print(f'saved metadata → {metadata_path}')

with report_path.open('w') as f:
    json.dump({
        'model': 'xgboost',
        'run_id': run_id,
        'git_commit': prov['git_commit'],
        'git_dirty': prov['git_dirty'],
        'calibration': CALIB_METHOD,
        'feature_set_version': prov['feature_set_version'],
        'features_sha256': prov['features_sha256'],
        'val': val_report.to_dict(),
        'test': test_report.to_dict(),
    }, f, indent=2)
print(f'saved report → {report_path}')

## 7. Hand-off

If XGBoost cleared the baseline on both PR-AUC and precision@10%, the model
in `data/models/xgb_<stamp>.joblib` is the **production estimator** for
notebook 06 — SHAP + scoring + `scores.parquet` + JSON export to the web app.

If XGBoost did NOT clear the baseline:
  - Stay with the baseline model in notebook 06.
  - Investigate: feature engineering miss? Categorical handling issue?
  - Don't add hyperparameter sweeps before checking the data — the
    typical fix is better features, not tuning.

**Next step**: `notebooks/06_eval_and_shap.ipynb` —
  - Per-restaurant SHAP top-drivers
  - Group-performance table by `static_facility_type` and `static_zip3`
  - Write `data/predictions/scores.parquet` and `app/public/data/scores.json`
    (the web app auto-drops the demo banner when the real one lands)

**Sanity to confirm before moving on**:
  - XGB beats baseline on PR-AUC AND precision@10%, OR a decision has been
    documented to ship baseline instead
  - Calibration curve is roughly on the diagonal
  - Top-importance feature list reads sensibly (`prior_*` should dominate;
    if `static_zip` is #1 you may have a leak you didn't realise)
  - `tests/` still all green